In [5]:
# ============================================================
# Federated Continual Learning Methods for Time-Series Forecasting
# ============================================================
# This script defines the experimental components used for task-free
# federated continual learning in time-series forecasting: reproducibility
# utilities, the LSTM forecaster, replay buffers, robust drift-threshold
# calibration, base federated training, and several continual-learning
# adaptation strategies.
# ============================================================

import os
import random
import time
from copy import deepcopy
from collections import defaultdict, deque

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.cluster import KMeans


# ============================================================
# 1. Reproducibility
# ============================================================
# Controls the random-number generators and backend settings needed
# for reproducible experimental runs.

def set_all_seeds(seed: int = 42):
    # Seed Python, NumPy, and PyTorch to reduce run-to-run variability.
    os.environ["PYTHONHASHSEED"] = str(seed)
    # Python hash seeding helps make hash-based operations deterministic.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Disable cuDNN benchmarking because it may choose nondeterministic kernels.
    try:
        # Some PyTorch operations may still be nondeterministic on specific devices.
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


# ============================================================
# 2. Dataset Wrapper
# ============================================================
# Converts lagged time-series windows and targets into a standard
# PyTorch Dataset interface.

class TimeSeriesDataset(Dataset):
    # Lightweight wrapper around already-windowed arrays.
    def __init__(self, X, y):
        # Convert inputs once so each DataLoader batch is already a float tensor.
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# ============================================================
# 3. Forecasting Model
# ============================================================
# Defines the shared LSTM architecture used by all clients and methods.

class LSTMPredictor(nn.Module):
    # A compact sequence model used consistently across all experiments.
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int, num_layers: int = 1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # The model receives a batch of lagged windows and predicts the horizon.
        # x: (B, seq_len, input_dim)
        # The last LSTM output summarizes the input window.
        lstm_out, _ = self.lstm(x)
        last_hidden = lstm_out[:, -1, :]
        return self.fc(last_hidden)


# ============================================================
# 4. Replay Buffers
# ============================================================
# Defines the memory structures used to store base and stream samples
# for replay-based and non-replay continual-learning methods.

class ReplayBuffer:
    # Simple list-backed FIFO memory used for representative base samples.
    """
    Basic FIFO replay buffer used to initialize representative base memory.
    """
    def __init__(self, capacity: int):
        self.capacity = int(capacity)
        self.buffer = []

    def add(self, x, y):
        # Store detached CPU tensors so replay does not keep computation graphs.
        if self.capacity <= 0:
            return
        if len(self.buffer) >= self.capacity:
            self.buffer.pop(0)
        self.buffer.append((x.detach().cpu(), y.detach().cpu()))

    def sample(self, batch_size: int):
        # Uniformly sample a replay mini-batch without replacement.
        if len(self.buffer) == 0:
            return None, None
        idx = np.random.choice(len(self.buffer), min(batch_size, len(self.buffer)), replace=False)
        X, y = zip(*[self.buffer[i] for i in idx])
        return torch.stack(X), torch.stack(y)

    def sample_subset(self, subset_size: int):
        # Return a list of pairs so the caller can batch them later.
        total = len(self.buffer)
        if total == 0:
            return []
        subset_size = min(int(subset_size), total)
        idx = np.random.choice(total, size=subset_size, replace=False)
        return [self.buffer[i] for i in idx]

    def __len__(self):
        return len(self.buffer)


class NormalReplayBuffer:
    # Single FIFO memory used by the standard replay baseline.
    """
    Standard single-memory replay buffer.

    Used for Normal Replay:
    - initialized with representative base samples,
    - streaming samples are inserted into the same unified FIFO memory,
    - no offline/online separation.
    """
    def __init__(self, capacity: int):
        self.capacity = int(capacity)
        self.buffer = deque(maxlen=self.capacity)
        # maxlen implements automatic removal of the oldest samples.

    def add(self, x, y):
        if self.capacity <= 0:
            return
        self.buffer.append((x.detach().cpu(), y.detach().cpu()))

    def add_batch(self, pairs):
        for x, y in pairs:
            self.add(x, y)

    def sample_subset(self, subset_size: int):
        total = len(self.buffer)
        if total == 0:
            return []
        subset_size = min(int(subset_size), total)
        if subset_size <= 0:
            return []
        idx = np.random.choice(total, size=subset_size, replace=False)
        buffer_list = list(self.buffer)
        return [buffer_list[i] for i in idx]

    def get_all_pairs(self):
        return list(self.buffer)

    def __len__(self):
        return len(self.buffer)

class TwoTierReplayBuffer:
    # Memory used by the proposed Two-Tier Replay strategy.
    """
    Two-zone replay buffer:
      - offline_fixed: frozen representative base samples;
      - online_fifo: recent streaming samples.
    """
    def __init__(self, offline_pairs, total_capacity: int, online_capacity: int = None):
        max_total_capacity = int(total_capacity)

        self.offline_fixed = [
            (x.detach().cpu(), y.detach().cpu())
            for (x, y) in offline_pairs
        ]

        self.n_offline = len(self.offline_fixed)

        if self.n_offline > max_total_capacity:
            self.offline_fixed = self.offline_fixed[:max_total_capacity]
            self.n_offline = len(self.offline_fixed)

        if online_capacity is None:
            self.online_capacity = max(0, max_total_capacity - self.n_offline)
        else:
            self.online_capacity = int(online_capacity)

        self.online_fifo = deque(maxlen=self.online_capacity)

        self.total_capacity = self.n_offline + self.online_capacity

        self._ver = 0
        self._cached = {"ver": -1, "X": None, "y": None}

    @property
    def capacity(self):
        return self.total_capacity

    @property
    def buffer(self):
        return list(self.offline_fixed) + list(self.online_fifo)

    def __len__(self):
        return self.n_offline + len(self.online_fifo)

    def __iter__(self):
        # Iterate over fixed offline samples first, then recent online samples.
        for p in self.offline_fixed:
            yield p
        for p in self.online_fifo:
            yield p

    def _bump_ver(self):
        # Mark cached tensorized views as stale after online-memory updates.
        self._ver += 1
        self._cached = {"ver": -1, "X": None, "y": None}

    def add(self, x, y):
        if self.online_capacity <= 0:
            return
        self.online_fifo.append((x.detach().cpu(), y.detach().cpu()))
        self._bump_ver()

    def add_batch_online(self, pairs):
        # Insert stream samples into the online FIFO tier only.
        if self.online_capacity <= 0:
            return
        for x, y in pairs:
            self.online_fifo.append((x.detach().cpu(), y.detach().cpu()))
        self._bump_ver()

    def sample(self, batch_size: int, p_offline: float = 0.5):
        # Draw a mixed mini-batch from offline and online memory.
        total = len(self)
        if total == 0:
            return None, None

        k = min(batch_size, total)
        n_off = min(int(round(p_offline * k)), self.n_offline)
        n_on = k - n_off

        if n_on > len(self.online_fifo):
            deficit = n_on - len(self.online_fifo)
            n_on = len(self.online_fifo)
            n_off = min(self.n_offline, n_off + deficit)

        rng = np.random.default_rng()
        Xs, Ys = [], []

        if n_off > 0:
            idx_off = rng.choice(self.n_offline, size=n_off, replace=False)
            for i in idx_off:
                x, y = self.offline_fixed[i]
                Xs.append(x)
                Ys.append(y)

        if n_on > 0:
            online_list = list(self.online_fifo)
            idx_on = rng.choice(len(online_list), size=n_on, replace=False)
            for i in idx_on:
                x, y = online_list[i]
                Xs.append(x)
                Ys.append(y)

        if len(Xs) == 0:
            return None, None
        return torch.stack(Xs), torch.stack(Ys)

    def sample_subset(self, subset_size: int, p_offline: float = 0.5):
        # Select a replay subset with an explicit offline/online sampling ratio.
        total = len(self)
        if total == 0:
            return []

        subset_size = min(int(subset_size), total)
        if subset_size <= 0:
            return []

        n_off = min(int(round(p_offline * subset_size)), self.n_offline)
        n_on = subset_size - n_off
        online_n = len(self.online_fifo)

        if n_on > online_n:
            deficit = n_on - online_n
            n_on = online_n
            n_off = min(self.n_offline, n_off + deficit)

        if n_off > self.n_offline:
            deficit = n_off - self.n_offline
            n_off = self.n_offline
            n_on = min(online_n, n_on + deficit)

        rng = np.random.default_rng()
        subset = []

        if n_off > 0:
            idx_off = rng.choice(self.n_offline, size=n_off, replace=False)
            subset.extend([self.offline_fixed[i] for i in idx_off])

        if n_on > 0:
            online_list = list(self.online_fifo)
            idx_on = rng.choice(len(online_list), size=n_on, replace=False)
            subset.extend([online_list[i] for i in idx_on])

        return subset


class RecentStreamBuffer:
    # Stores recent stream samples for methods without replay memory.
    """
    Stores only recent revealed stream samples.
    Used for Naive, EWC, LwF, and SI baselines.
    """
    def __init__(self, max_samples: int):
        self.max_samples = int(max_samples)
        self.buffer = deque(maxlen=self.max_samples)

    def add(self, x, y):
        self.buffer.append((x.detach().cpu(), y.detach().cpu()))

    def add_batch(self, pairs):
        for x, y in pairs:
            self.add(x, y)

    def get_all_pairs(self):
        return list(self.buffer)

    def sample_all(self):
        if len(self.buffer) == 0:
            return None, None
        X = torch.stack([p[0] for p in self.buffer])
        y = torch.stack([p[1] for p in self.buffer])
        return X, y

    def clear(self):
        self.buffer.clear()

    def __len__(self):
        return len(self.buffer)


# ============================================================
# 5. Buffer Builders
# ============================================================
# Creates and initializes replay memories from the offline/base period.

def init_base_buffer(predictor, loader, device, buffer_capacity: int, sample_fraction: float = 0.05):
    # Select representative base-period samples using KMeans centroids.
    """
    KMeans-based representative sample selection from base data.
    """
    predictor.eval()
    all_X, all_y = [], []

    with torch.no_grad():
        for X, y in loader:
            all_X.append(X.cpu())
            all_y.append(y.cpu())

    if len(all_X) == 0:
        return ReplayBuffer(capacity=buffer_capacity)

    all_X = torch.cat(all_X, dim=0)
    all_y = torch.cat(all_y, dim=0)
    N = all_X.size(0)

    n_selected = max(1, int(round(sample_fraction * N)))
    n_selected = min(n_selected, N, buffer_capacity)

    X_embed = all_X.view(N, -1).numpy()
    # Flatten each time-series window so clustering is performed in vector space.

    kmeans = KMeans(n_clusters=n_selected, n_init="auto", random_state=42)
    labels = kmeans.fit_predict(X_embed)
    centers = kmeans.cluster_centers_

    selected_indices = []
    for k in range(n_selected):
        # Use the observed sample nearest to each centroid as the representative.
        cluster_idx = np.where(labels == k)[0]
        if cluster_idx.size == 0:
            continue
        dists = np.linalg.norm(X_embed[cluster_idx] - centers[k], axis=1)
        closest = cluster_idx[np.argmin(dists)]
        selected_indices.append(int(closest))

    buffer = ReplayBuffer(capacity=buffer_capacity)
    for i in selected_indices[:buffer_capacity]:
        buffer.add(all_X[i], all_y[i])

    return buffer

def upgrade_to_two_tier(
    base_buffer: ReplayBuffer,
    total_capacity: int,
    online_capacity: int = None
):
    offline_pairs = list(base_buffer.buffer)

    return TwoTierReplayBuffer(
        offline_pairs=offline_pairs,
        total_capacity=total_capacity,
        online_capacity=online_capacity
    )

def build_normal_replay_buffers_from_base(initial_buffers: dict, capacity: int):
    # Initialize normal replay from the same base samples used by other replay methods.
    """
    Build standard replay buffers initialized only from base representative samples.
    If input is TwoTierReplayBuffer, use offline_fixed only.
    """
    normal_buffers = {}

    for client, buf in initial_buffers.items():
        normal_buf = NormalReplayBuffer(capacity=capacity)

        if hasattr(buf, "offline_fixed"):
            base_pairs = list(buf.offline_fixed)
        elif hasattr(buf, "buffer"):
            base_pairs = list(buf.buffer)
        else:
            base_pairs = list(buf)

        normal_buf.add_batch(base_pairs)
        normal_buffers[client] = normal_buf

    return normal_buffers


def reset_online_zone(buffers: dict):
    # Clear only online stream memory; fixed offline memory remains intact.
    for _, buf in buffers.items():
        if hasattr(buf, "online_fifo"):
            buf.online_fifo.clear()
            if hasattr(buf, "_bump_ver"):
                buf._bump_ver()


def prefill_online_with_calibration_windows(task_lagged_base_test: dict, buffers: dict, warmup_windows: int, pred_len: int):
    # Optionally seed online memory with warm-up stream windows.
    if warmup_windows <= 0:
        return

    T = warmup_windows * pred_len

    for client, data in task_lagged_base_test.items():
        buf = buffers.get(client, None)
        if buf is None:
            continue

        upper = min(T, len(data["X"]), len(data["y"]))
        pending = []

        for idx in range(upper):
            x_i = torch.tensor(data["X"][idx], dtype=torch.float32)
            y_i = torch.tensor(data["y"][idx], dtype=torch.float32)
            pending.append((x_i, y_i))

        if hasattr(buf, "add_batch_online"):
            buf.add_batch_online(pending)
        elif hasattr(buf, "add_batch"):
            buf.add_batch(pending)
        elif hasattr(buf, "add"):
            for x_i, y_i in pending:
                buf.add(x_i, y_i)


# ============================================================
# 6. General Helpers
# ============================================================
# Provides shared routines for RMSE evaluation, FedAvg aggregation,
# recent-window construction, and mini-batch sampling.

@torch.no_grad()
def _predict_rmse(model, x_np, y_np, device):
    # Evaluate one lagged window and return its RMSE.
    model.eval()
    x = torch.tensor(x_np, dtype=torch.float32, device=device).unsqueeze(0)
    # Add the batch dimension expected by the LSTM model.
    y_hat = model(x).detach().cpu().numpy().ravel()
    y_true = y_np.astype(np.float32).ravel()
    if y_hat.shape != y_true.shape:
        return np.nan
    return float(np.sqrt(np.mean((y_hat - y_true) ** 2)))


def _weighted_fedavg_state_dicts(state_dicts, weights):
    # Aggregate local state_dict objects using sample-count weights.
    if len(state_dicts) == 0:
        raise ValueError("Cannot aggregate an empty list of state_dicts.")

    total_weight = float(sum(weights)) if sum(weights) > 0 else 1.0
    avg_state = {k: torch.zeros_like(v) for k, v in state_dicts[0].items()}

    for sd, w in zip(state_dicts, weights):
        # Accumulate a weighted average for every parameter tensor.
        coeff = float(w) / total_weight
        for k in avg_state:
            avg_state[k] += sd[k].detach().cpu() * coeff

    return avg_state


def _make_recent_pairs(X_full, Y_full, t: int, pred_len: int, persistence_M: int):
    # Build the recent stream window associated with persistent drift.
    recent_span = pred_len * max(1, int(persistence_M))
    start_idx = max(0, t - recent_span + 1)
    end_idx = min(len(X_full), len(Y_full), t + pred_len)

    recent_pairs = []
    for idx in range(start_idx, end_idx):
        x_i = torch.tensor(X_full[idx], dtype=torch.float32)
        y_i = torch.tensor(Y_full[idx], dtype=torch.float32)
        recent_pairs.append((x_i, y_i))

    return recent_pairs


def _make_current_chunk_pairs(X_full, Y_full, t: int, pred_len: int):
    # Build the currently revealed prediction chunk.
    upper = min(t + pred_len, len(X_full), len(Y_full))
    pairs = []
    for idx in range(t, upper):
        x_i = torch.tensor(X_full[idx], dtype=torch.float32)
        y_i = torch.tensor(Y_full[idx], dtype=torch.float32)
        pairs.append((x_i, y_i))
    return pairs


def _add_current_chunk_to_online_buffer(buffer, X_full, Y_full, t: int, pred_len: int):
    # Dispatch to the appropriate buffer method while keeping one shared interface.
    pending = _make_current_chunk_pairs(X_full, Y_full, t, pred_len)
    if hasattr(buffer, "add_batch_online"):
        buffer.add_batch_online(pending)
    elif hasattr(buffer, "add_batch"):
        buffer.add_batch(pending)
    elif hasattr(buffer, "add"):
        for x_i, y_i in pending:
            buffer.add(x_i, y_i)


def iterate_subset_batches(pairs, batch_size: int, device: str = None, shuffle: bool = False):
    # Yield mini-batches from a list of stored (x, y) pairs.
    if pairs is None or len(pairs) == 0:
        return

    idx = np.arange(len(pairs))
    # The shuffle argument is retained for interface compatibility.


    for start in range(0, len(idx), batch_size):
        batch_idx = idx[start:start + batch_size]
        Xb = torch.stack([pairs[i][0] for i in batch_idx])
        Yb = torch.stack([pairs[i][1] for i in batch_idx])

        if device is not None:
            Xb = Xb.to(device, non_blocking=True)
            Yb = Yb.to(device, non_blocking=True)

        yield Xb, Yb

def sample_pairs_batch(pairs, batch_size: int, device: str):
    # Randomly sample one mini-batch from a stored list of pairs.
    """
    Randomly sample one mini-batch from a list of (x, y) pairs.
    Used to balance current/recent stream samples and replay samples.
    """
    if pairs is None or len(pairs) == 0:
        return None, None

    b = min(batch_size, len(pairs))
    idx = np.random.choice(len(pairs), size=b, replace=False)

    Xb = torch.stack([pairs[i][0] for i in idx]).to(device)
    Yb = torch.stack([pairs[i][1] for i in idx]).to(device)

    return Xb, Yb        
        

def _clone_named_params(model, device):
    # Snapshot parameters by name for regularization-based methods.
    return {n: p.detach().clone().to(device) for n, p in model.named_parameters()}


def _zeros_like_named(params_dict):
    # Create zero accumulators with the same shapes as a parameter snapshot.
    return {n: torch.zeros_like(t) for n, t in params_dict.items()}


# ============================================================
# 7. Evaluation Helpers
# ============================================================
# Computes base/test and stream RMSE summaries without updating models.

@torch.no_grad()
def evaluate_models_on_dataloaders(models_by_client, dataloaders_by_client, device="cpu"):
    # Evaluate each available client model on its corresponding dataloader.
    per_client_rmse = {}

    for client, loader in dataloaders_by_client.items():
        if client not in models_by_client:
            continue

        model = models_by_client[client].to(device).eval()
        se_sum, n = 0.0, 0

        for X, y in loader:
            X, y = X.to(device), y.to(device)
            y_hat = model(X)
            se_sum += torch.sum((y_hat - y) ** 2).item()
            n += y.numel()

        per_client_rmse[client] = float(np.sqrt(se_sum / max(1, n)))

    valid = [v for v in per_client_rmse.values() if np.isfinite(v)]
    avg_rmse = float(np.mean(valid)) if valid else float("nan")
    return per_client_rmse, avg_rmse


@torch.no_grad()
def evaluate_static_stream_rmse(models_by_client, task_lagged_base_test, device, pred_len, warmup_windows):
    # Measure the stream performance of the non-adaptive static baseline.
    stream_rmses = {}

    min_len = min(len(v["y"]) for v in task_lagged_base_test.values())
    max_chunks = max(0, min_len // pred_len)
    start_chunk = warmup_windows

    for client, data in task_lagged_base_test.items():
        model = models_by_client[client].to(device).eval()
        X_full, Y_full = data["X"], data["y"]
        rmses = []

        for chunk_id in range(start_chunk, max_chunks):
            t = chunk_id * pred_len
            if t >= len(Y_full):
                break
            rmse = _predict_rmse(model, X_full[t], Y_full[t], device)
            if np.isfinite(rmse):
                rmses.append(rmse)

        stream_rmses[client] = rmses

    return {
        "stream_rmses": stream_rmses,
        "final_models": models_by_client,
        "drift_counts": {c: 0 for c in task_lagged_base_test.keys()},
        "total_drifts": 0,
        "avg_drifts_per_client": 0.0,
        "global_drift_events": 0,
        "client_predictions": {},
    }


def mean_stream_rmse(stream_rmses):
    # Collapse per-client stream RMSE traces into one average score.
    vals = []
    for _, rmses in stream_rmses.items():
        if isinstance(rmses, list) and len(rmses) > 0:
            vals.append(float(np.mean(rmses)))
        elif isinstance(rmses, (int, float, np.float32, np.float64)):
            vals.append(float(rmses))
    return float(np.mean(vals)) if vals else float("nan")


# ============================================================
# 8. Drift Threshold Calibration
# ============================================================
# Calibrates robust per-client RMSE thresholds from base-period errors
# using the median and median absolute deviation.

@torch.no_grad()
def calibrate_theta_mad_from_base(base_lagged: dict, models_by_client: dict, *, device="cpu", k: float = 1.0, min_mad: float = 1e-6, verbose: bool = True):
    # Compute a robust client-specific threshold theta_c = median + k * scaled_MAD.
    theta = {}
    baseline_stats = {}

    for client, data in base_lagged.items():
        model = models_by_client.get(client, None)
        if model is None:
            continue

        model = model.to(device).eval()
        X_full, Y_full = data["X"], data["y"]
        if len(X_full) == 0 or len(Y_full) == 0:
            continue

        rmses = []
        for i in range(len(Y_full)):
            rmse_i = _predict_rmse(model, X_full[i], Y_full[i], device)
            if np.isfinite(rmse_i):
                rmses.append(rmse_i)

        if len(rmses) == 0:
            continue

        rmses = np.asarray(rmses, dtype=np.float64)
        med = float(np.median(rmses))
        mad = float(np.median(np.abs(rmses - med)))
        scaled_mad = float(1.4826 * max(mad, min_mad))
        # 1.4826 rescales MAD to approximate a standard deviation under normality.
        theta_c = float(med + k * scaled_mad)

        theta[client] = theta_c
        baseline_stats[client] = {
            "median_rmse": med,
            "mad_rmse": mad,
            "scaled_mad": scaled_mad,
            "theta": theta_c,
            "n_windows": int(len(rmses)),
        }

    if verbose:
        print("\nRobust threshold calibration from base data")
        for client, stats in baseline_stats.items():
            print(
                f"{client:<15} | median={stats['median_rmse']:.6f} | "
                f"MAD={stats['mad_rmse']:.6f} | theta={stats['theta']:.6f} | "
                f"n={stats['n_windows']}"
            )

    return theta, baseline_stats


# ============================================================
# 9. EWC / SI / LwF Helper Functions
# ============================================================
# Implements the regularization and distillation losses required by
# EWC, Synaptic Intelligence, and Learning without Forgetting.

def compute_ewc_fisher(model, dataloader, device="cpu", num_batches=10):
    # Approximate the diagonal Fisher information using squared gradients.
    model.eval().to(device)
    for p in model.parameters():
        p.requires_grad = True

    fisher = {n: torch.zeros_like(p, device=device) for n, p in model.named_parameters()}
    count = 0

    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        model.zero_grad(set_to_none=True)
        loss = F.mse_loss(model(X), y, reduction="mean")
        loss.backward()

        for n, p in model.named_parameters():
            if p.grad is not None:
                fisher[n] += p.grad.detach().pow(2)

        count += 1
        if num_batches is not None and count >= num_batches:
            break

    for n in fisher:
        fisher[n] /= max(1, count)

    return fisher


def compute_ewc_penalty(model, fisher_dict, ref_params_dict):
    # Penalize deviations from base reference parameters using Fisher weights.
    penalty = 0.0
    for n, p in model.named_parameters():
        if n in fisher_dict and n in ref_params_dict:
            penalty = penalty + (fisher_dict[n] * (p - ref_params_dict[n]).pow(2)).sum()
    return penalty

def compute_si_penalty(model, omega_dict, ref_params_dict):
    # Penalize deviations from SI reference parameters using omega weights.
    device = next(model.parameters()).device
    penalty = torch.zeros((), device=device)

    for n, p in model.named_parameters():
        if n in omega_dict and n in ref_params_dict:
            omega = torch.clamp(omega_dict[n].to(device), min=0.0)
            ref = ref_params_dict[n].to(device)
            penalty = penalty + (omega * (p - ref).pow(2)).sum()

    return penalty

def compute_lwf_kd_loss(student_preds, teacher_preds):
    # Mean-squared prediction matching for teacher-student distillation.
    return F.mse_loss(student_preds, teacher_preds)


# ============================================================
# 10. Federated Base Training with SI Artifact Capture
# ============================================================
# Trains the initial FedAvg model and records the quantities needed
# to later evaluate the SI baseline fairly.

def train_federated_base_model(
    client_dataloaders,
    input_dim,
    output_dim,
    num_rounds=100,
    local_epochs=1,
    hidden_dim=64,
    lr=1e-3,
    device="cpu",
    eval_every=1,
    *,
    si_eps: float = 1e-3,
):
    predictors = {}
    history = {"round": [], "rmse_base_test_avg": []}

    global_model = LSTMPredictor(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim).to(device)
    # Initialize the global model distributed to all clients in round 1.
    global_weights = {k: v.detach().clone() for k, v in global_model.state_dict().items()}

    si_theta_init = {}
    # SI stores each client's initial parameters and path-integral accumulators.
    si_W_accum = {}

    for rnd in range(1, num_rounds + 1):
        print(f"\n[BASE FL] Round {rnd}/{num_rounds}")
        local_states, local_sizes = [], []

        for client, loader in client_dataloaders["base_train"].items():
            # Each client starts local training from the current global weights.
            model = LSTMPredictor(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim).to(device)
            model.load_state_dict(global_weights)
            model.train()
            opt = torch.optim.Adam(model.parameters(), lr=lr)

            if client not in si_theta_init:
                si_theta_init[client] = _clone_named_params(model, device)
                si_W_accum[client] = _zeros_like_named(si_theta_init[client])

            prev_params = _clone_named_params(model, device)

            for _ in range(local_epochs):
                for X, y in loader:
                    X, y = X.to(device), y.to(device)
                    opt.zero_grad(set_to_none=True)
                    loss = F.mse_loss(model(X), y)
                    loss.backward()

                    grads = {
                        # SI importance accumulation uses gradients and parameter displacement.
                        n: (p.grad.detach().clone() if p.grad is not None else torch.zeros_like(p))
                        for n, p in model.named_parameters()
                    }

                    opt.step()

                    with torch.no_grad():
                        for n, p in model.named_parameters():
                            delta = p.detach() - prev_params[n]
                            si_W_accum[client][n] += (-grads[n]) * delta
                            prev_params[n] = p.detach().clone()
                            

            local_states.append({k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
            local_sizes.append(len(loader.dataset))
            predictors[client] = model.eval()

        global_weights_cpu = _weighted_fedavg_state_dicts(local_states, local_sizes)
        # Server step: aggregate client models by weighted FedAvg.
        global_weights = {k: v.to(device) for k, v in global_weights_cpu.items()}

    base_predictor = LSTMPredictor(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim).to(device)
    # Materialize the final global model after base training.
    base_predictor.load_state_dict(global_weights)
    base_predictor.eval()
    base_predictors = {}

    for client in client_dataloaders["base_train"].keys():
        m = LSTMPredictor(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            output_dim=output_dim
        ).to(device)

        m.load_state_dict(global_weights)
        m.eval()

        base_predictors[client] = m    
    # ------------------------------------------------------------
    # SI reference and omega construction
    # ------------------------------------------------------------
    si_ref_params_by_client = {}
    si_omegas_by_client = {}

    omega_clip = 10.0

    for client in client_dataloaders["base_train"].keys():

        # Since CL starts from the final global FedAvg model,
        # SI should use the same model as its reference.
        theta_ref = _clone_named_params(base_predictors[client].to(device).eval(), device)

        theta_init = si_theta_init[client]
        W = si_W_accum[client]

        # IMPORTANT: this must be inside the client loop
        omegas = {}

        with torch.no_grad():
            for n in theta_ref.keys():
                delta_total = theta_ref[n] - theta_init[n]
                denom = delta_total.pow(2) + si_eps

                omega_n = W[n] / denom
                omegas[n] = torch.clamp(
                    omega_n,
                    min=0.0,
                    max=omega_clip
                ).detach().clone()

        si_ref_params_by_client[client] = theta_ref
        si_omegas_by_client[client] = omegas


    return base_predictor, base_predictors, history, si_ref_params_by_client, si_omegas_by_client


# ============================================================
# 11. Shared Streaming Loop Utilities
# ============================================================
# Centralizes the prediction, drift-detection, buffer-update, and
# result-formatting logic used by all streaming methods.

def _initialize_stream_state(task_lagged_base_test, personalized_predictors, device, clone_models=True):
    # Prepare model copies, counters, RMSE logs, and prediction traces.
    models = {
        c: deepcopy(m).to(device).eval()
        for c, m in personalized_predictors.items()
    } if clone_models else personalized_predictors

    drift_counts = {c: 0 for c in task_lagged_base_test.keys()}
    exceed_count = {c: 0 for c in task_lagged_base_test.keys()}
    stream_rmses = defaultdict(list)
    client_predictions = {c: {"y_pred": [], "y_true": []} for c in task_lagged_base_test.keys()}

    return models, drift_counts, exceed_count, stream_rmses, client_predictions


def _stream_predict_detect_update_buffers(
    *,
    models,
    task_lagged_base_test,
    device,
    pred_len,
    t,
    theta,
    rmse_threshold_fallback,
    persistence_M,
    drift_counts,
    exceed_count,
    stream_rmses,
    client_predictions,
    buffer_update_fn=None,
):
    drifting_clients = []

    for client, data in task_lagged_base_test.items():
        model = models[client].to(device).eval()
        X_full, Y_full = data["X"], data["y"]

        if t >= len(Y_full):
            continue

        rmse = _predict_rmse(model, X_full[t], Y_full[t], device)
        # Evaluate before any adaptation on the current stream chunk.
        stream_rmses[client].append(rmse)

        x_tensor = torch.tensor(X_full[t], dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            y_pred = model(x_tensor).detach().cpu().numpy().ravel()
        y_true = Y_full[t].astype(np.float32).ravel()
        client_predictions[client]["y_pred"].append(y_pred)
        client_predictions[client]["y_true"].append(y_true)

        if buffer_update_fn is not None:
            # After the current target is revealed, update the method-specific memory.
            buffer_update_fn(client, X_full, Y_full, t, pred_len)

        threshold = theta.get(client, rmse_threshold_fallback)
        # Drift is confirmed only after persistent threshold exceedance.
        exceed_count[client] = exceed_count[client] + 1 if rmse > threshold else 0

        if exceed_count[client] >= max(1, int(persistence_M)):
            drifting_clients.append(client)
            drift_counts[client] += 1

    return drifting_clients


def _finalize_result(models, buffers, drift_counts, global_drift_events, drift_history, stream_rmses, client_predictions):
    # Return a consistent result dictionary for all methods.
    return {
        "stream_rmses": dict(stream_rmses),
        "final_models": models,
        "final_buffers": buffers,
        "drift_counts": drift_counts,
        "total_drifts": int(sum(drift_counts.values())),
        "avg_drifts_per_client": float(np.mean(list(drift_counts.values()))) if len(drift_counts) > 0 else 0.0,
        "global_drift_events": global_drift_events,
        "drift_history": drift_history,
        "client_predictions": client_predictions,
        "overall_stream_rmse": mean_stream_rmse(stream_rmses),
    }


# ============================================================
# 12. Method 1: Naive Continual Learning
# ============================================================
# Updates only on recent stream samples after drift is detected.
# This baseline has plasticity but no explicit anti-forgetting mechanism.

def NaiveContinualLearning(
    *,
    task_lagged_base_test: dict,
    personalized_predictors: dict,
    device: str = "cpu",
    pred_len: int = 6,
    theta_override: dict = None,
    warmup_windows: int = 5,
    rmse_threshold_fallback: float = 0.22,
    persistence_M: int = 2,
    recent_buffer_capacity: int = None,
    R: int = 5,
    E: int = 1,
    lr: float = 1e-2,
    batch_size: int = 32,
    adapt_all_clients_on_global_drift: bool = True,
    clone_models: bool = True,
):
    print("\nRunning Naive Continual Learning...")

    models, drift_counts, exceed_count, stream_rmses, client_predictions = _initialize_stream_state(
        task_lagged_base_test, personalized_predictors, device, clone_models
    )

    if recent_buffer_capacity is None:
        # By default, store exactly the recent span used for persistence detection.
        recent_buffer_capacity = persistence_M * pred_len

    recent_buffers = {
        c: RecentStreamBuffer(max_samples=recent_buffer_capacity)
        for c in task_lagged_base_test.keys()
    }

    theta = dict(theta_override) if theta_override is not None else {}
    min_len = min(len(v["y"]) for v in task_lagged_base_test.values())
    max_chunks = max(0, min_len // pred_len)
    start_chunk = warmup_windows

    global_drift_events = 0
    drift_history = []

    if start_chunk >= max_chunks:
        return _finalize_result(models, recent_buffers, drift_counts, 0, [], stream_rmses, client_predictions)

    for chunk_id in range(start_chunk, max_chunks):
        t = chunk_id * pred_len

        def update_recent(client, X_full, Y_full, t0, p):
            recent_buffers[client].add_batch(_make_current_chunk_pairs(X_full, Y_full, t0, p))

        drifting_clients = _stream_predict_detect_update_buffers(
            models=models,
            task_lagged_base_test=task_lagged_base_test,
            device=device,
            pred_len=pred_len,
            t=t,
            theta=theta,
            rmse_threshold_fallback=rmse_threshold_fallback,
            persistence_M=persistence_M,
            drift_counts=drift_counts,
            exceed_count=exceed_count,
            stream_rmses=stream_rmses,
            client_predictions=client_predictions,
            buffer_update_fn=update_recent,
        )

        if len(drifting_clients) == 0:
            # No detected drift means no adaptation for this stream chunk.
            continue

        global_drift_events += 1
        drift_history.append({"chunk_id": chunk_id, "t": t, "drifting_clients": drifting_clients})
        participating_clients = list(task_lagged_base_test.keys()) if adapt_all_clients_on_global_drift else drifting_clients

        for _round in range(R):
            local_states, local_weights = [], []

            for client in participating_clients:
                pairs = recent_buffers[client].get_all_pairs()
                if len(pairs) == 0:
                    continue

                local_model = deepcopy(models[client]).to(device).train()
                optimizer = torch.optim.Adam(local_model.parameters(), lr=lr)

                for _ in range(E):
                    for Xb, Yb in iterate_subset_batches(pairs, batch_size=batch_size, device=device, shuffle=False):
                        optimizer.zero_grad(set_to_none=True)
                        loss = F.mse_loss(local_model(Xb), Yb)
                        loss.backward()
                        optimizer.step()

                local_states.append({k: v.detach().cpu().clone() for k, v in local_model.state_dict().items()})
                local_weights.append(len(pairs))

            if len(local_states) > 0:
                # Synchronize client models to the aggregated adapted model.
                avg_state = _weighted_fedavg_state_dicts(local_states, local_weights)
                for client in models:
                    models[client].load_state_dict(avg_state)
                    models[client].to(device).eval()

        for client in exceed_count:
            # Reset persistence counters after an adaptation event.
            exceed_count[client] = 0

    return _finalize_result(models, recent_buffers, drift_counts, global_drift_events, drift_history, stream_rmses, client_predictions)


# ============================================================
# 13. Method 2: Normal Replay
# ============================================================
# Uses one unified FIFO replay memory containing base and stream samples.

def ContinualLearning_NormalReplay(
    *,
    task_lagged_base_test: dict,
    personalized_predictors: dict,
    initial_replay_buffers: dict,
    device: str = "cpu",
    pred_len: int = 6,
    theta_override: dict = None,
    warmup_windows: int = 5,
    rmse_threshold_fallback: float = 0.22,
    persistence_M: int = 2,
    R: int = 5,
    E: int = 1,
    lr: float = 1e-2,
    batch_size: int = 32,
    replay_fraction: float = 0.30,
    replay_min_samples: int = 32,
    replay_max_samples: int = 500,
    lambda_current: float = 1.0,
    lambda_replay: float = 1,
    adapt_all_clients_on_global_drift: bool = True,
    clone_models: bool = True,
):
    print("\nRunning Normal Replay Continual Learning...")

    models, drift_counts, exceed_count, stream_rmses, client_predictions = _initialize_stream_state(
        task_lagged_base_test, personalized_predictors, device, clone_models
    )
    buffers = {c: deepcopy(b) for c, b in initial_replay_buffers.items()}
    # Deep copies ensure that repeated method runs start from identical memories.
    theta = dict(theta_override) if theta_override is not None else {}

    min_len = min(len(v["y"]) for v in task_lagged_base_test.values())
    max_chunks = max(0, min_len // pred_len)
    start_chunk = warmup_windows
    global_drift_events = 0
    drift_history = []

    if start_chunk >= max_chunks:
        return _finalize_result(models, buffers, drift_counts, 0, [], stream_rmses, client_predictions)

    for chunk_id in range(start_chunk, max_chunks):
        t = chunk_id * pred_len

        def update_buffer(client, X_full, Y_full, t0, p):
            buffers[client].add_batch(_make_current_chunk_pairs(X_full, Y_full, t0, p))

        drifting_clients = _stream_predict_detect_update_buffers(
            models=models,
            task_lagged_base_test=task_lagged_base_test,
            device=device,
            pred_len=pred_len,
            t=t,
            theta=theta,
            rmse_threshold_fallback=rmse_threshold_fallback,
            persistence_M=persistence_M,
            drift_counts=drift_counts,
            exceed_count=exceed_count,
            stream_rmses=stream_rmses,
            client_predictions=client_predictions,
            buffer_update_fn=update_buffer,
        )

        if len(drifting_clients) == 0:
            continue

        global_drift_events += 1
        drift_history.append({"chunk_id": chunk_id, "t": t, "drifting_clients": drifting_clients})
        participating_clients = list(task_lagged_base_test.keys()) if adapt_all_clients_on_global_drift else drifting_clients

        for _round in range(R):
            local_states, local_weights = [], []

            for client in participating_clients:
                X_full, Y_full = task_lagged_base_test[client]["X"], task_lagged_base_test[client]["y"]
                recent_pairs = _make_recent_pairs(X_full, Y_full, t, pred_len, persistence_M)
                if len(recent_pairs) == 0 or len(buffers[client]) == 0:
                    continue

                subset_size = int(round(replay_fraction * len(buffers[client])))
                # Use a bounded replay subset instead of the full memory.
                subset_size = max(replay_min_samples, subset_size)
                subset_size = min(replay_max_samples, subset_size, len(buffers[client]))
                replay_pairs = buffers[client].sample_subset(subset_size)
                if len(replay_pairs) == 0:
                    continue

                local_model = deepcopy(models[client]).to(device).train()
                optimizer = torch.optim.Adam(local_model.parameters(), lr=lr)

                n_steps = max(
                    1,
                    int(np.ceil(max(len(recent_pairs), len(replay_pairs)) / batch_size))
                )

                for _ in range(E):
                    for _step in range(n_steps):
                        X_cur_b, Y_cur_b = sample_pairs_batch(
                            recent_pairs,
                            batch_size=batch_size,
                            device=device
                        )

                        X_rep_b, Y_rep_b = sample_pairs_batch(
                            replay_pairs,
                            batch_size=batch_size,
                            device=device
                        )

                        if X_cur_b is None or X_rep_b is None:
                            continue

                        optimizer.zero_grad(set_to_none=True)

                        loss_current = F.mse_loss(local_model(X_cur_b), Y_cur_b)
                        loss_replay = F.mse_loss(local_model(X_rep_b), Y_rep_b)

                        loss = lambda_current * loss_current + lambda_replay * loss_replay
                        # Balance current-stream plasticity with replay-based stability.
                        loss.backward()
                        optimizer.step()

                local_states.append({k: v.detach().cpu().clone() for k, v in local_model.state_dict().items()})
                local_weights.append(len(recent_pairs) + len(replay_pairs))

            if len(local_states) > 0:
                avg_state = _weighted_fedavg_state_dicts(local_states, local_weights)
                for client in models:
                    models[client].load_state_dict(avg_state)
                    models[client].to(device).eval()

        for client in exceed_count:
            exceed_count[client] = 0

    return _finalize_result(models, buffers, drift_counts, global_drift_events, drift_history, stream_rmses, client_predictions)


# ============================================================
# 14. Method 3: Two-Tier Replay
# ============================================================
# Uses a fixed offline tier for base samples and a separate online FIFO
# tier for recent stream samples.

def ContinualLearning_TwoTierReplay(
    *,
    task_lagged_base_test: dict,
    personalized_predictors: dict,
    initial_replay_buffers: dict,
    device: str = "cpu",
    pred_len: int = 6,
    theta_override: dict = None,
    warmup_windows: int = 5,
    rmse_threshold_fallback: float = 0.22,
    persistence_M: int = 2,
    R: int = 5,
    E: int = 1,
    lr: float = 1e-2,
    batch_size: int = 32,
    p_offline: float = 0.85,
    replay_fraction: float = 0.30,
    replay_min_samples: int = 32,
    replay_max_samples: int = 500,
    lambda_current: float = 1.0,
    lambda_replay: float = 1.0,
    adapt_all_clients_on_global_drift: bool = True,
    clone_models: bool = True,
):
    print("\nRunning Two-Tier Replay Continual Learning...")

    models, drift_counts, exceed_count, stream_rmses, client_predictions = _initialize_stream_state(
        task_lagged_base_test, personalized_predictors, device, clone_models
    )
    buffers = {c: deepcopy(b) for c, b in initial_replay_buffers.items()}
    theta = dict(theta_override) if theta_override is not None else {}

    min_len = min(len(v["y"]) for v in task_lagged_base_test.values())
    max_chunks = max(0, min_len // pred_len)
    start_chunk = warmup_windows
    global_drift_events = 0
    drift_history = []

    if start_chunk >= max_chunks:
        return _finalize_result(models, buffers, drift_counts, 0, [], stream_rmses, client_predictions)

    for chunk_id in range(start_chunk, max_chunks):
        t = chunk_id * pred_len

        def update_buffer(client, X_full, Y_full, t0, p):
            _add_current_chunk_to_online_buffer(buffers[client], X_full, Y_full, t0, p)

        drifting_clients = _stream_predict_detect_update_buffers(
            models=models,
            task_lagged_base_test=task_lagged_base_test,
            device=device,
            pred_len=pred_len,
            t=t,
            theta=theta,
            rmse_threshold_fallback=rmse_threshold_fallback,
            persistence_M=persistence_M,
            drift_counts=drift_counts,
            exceed_count=exceed_count,
            stream_rmses=stream_rmses,
            client_predictions=client_predictions,
            buffer_update_fn=update_buffer,
        )

        if len(drifting_clients) == 0:
            continue

        global_drift_events += 1
        drift_history.append({"chunk_id": chunk_id, "t": t, "drifting_clients": drifting_clients})
        participating_clients = list(task_lagged_base_test.keys()) if adapt_all_clients_on_global_drift else drifting_clients

        for _round in range(R):
            local_states, local_weights = [], []

            for client in participating_clients:
                X_full, Y_full = task_lagged_base_test[client]["X"], task_lagged_base_test[client]["y"]
                recent_pairs = _make_recent_pairs(X_full, Y_full, t, pred_len, persistence_M)
                if len(recent_pairs) == 0 or len(buffers[client]) == 0:
                    continue

                subset_size = int(round(replay_fraction * len(buffers[client])))
                subset_size = max(replay_min_samples, subset_size)
                subset_size = min(replay_max_samples, subset_size, len(buffers[client]))

                replay_pairs = buffers[client].sample_subset(subset_size=subset_size, p_offline=p_offline)
                # Explicitly control the fraction of replay drawn from the fixed offline tier.
                if len(replay_pairs) == 0:
                    continue

                local_model = deepcopy(models[client]).to(device).train()
                optimizer = torch.optim.Adam(local_model.parameters(), lr=lr)

                X_cur, Y_cur = zip(*recent_pairs)
                # Current/recent samples form the plasticity term in the objective.
                X_cur = torch.stack(X_cur).to(device)
                Y_cur = torch.stack(Y_cur).to(device)

                for _ in range(E):
                    for Xb, Yb in iterate_subset_batches(replay_pairs, batch_size=batch_size, device=device, shuffle=False):
                        optimizer.zero_grad(set_to_none=True)
                        loss_current = F.mse_loss(local_model(X_cur), Y_cur)
                        loss_replay = F.mse_loss(local_model(Xb), Yb)
                        loss = lambda_current * loss_current + lambda_replay * loss_replay
                        loss.backward()
                        optimizer.step()

                local_states.append({k: v.detach().cpu().clone() for k, v in local_model.state_dict().items()})
                local_weights.append(len(recent_pairs) + len(replay_pairs))

            if len(local_states) > 0:
                avg_state = _weighted_fedavg_state_dicts(local_states, local_weights)
                for client in models:
                    models[client].load_state_dict(avg_state)
                    models[client].to(device).eval()

        for client in exceed_count:
            exceed_count[client] = 0

    return _finalize_result(models, buffers, drift_counts, global_drift_events, drift_history, stream_rmses, client_predictions)


# ============================================================
# 15. Method 4: EWC
# ============================================================
# Penalizes movement of parameters that are important for the base task.

def EWCContinualLearning(
    *,
    task_lagged_base_test: dict,
    personalized_predictors: dict,
    ewc_fishers_by_client: dict,
    ewc_ref_params_by_client: dict,
    device: str = "cpu",
    pred_len: int = 6,
    theta_override: dict = None,
    warmup_windows: int = 5,
    rmse_threshold_fallback: float = 0.22,
    persistence_M: int = 2,
    recent_buffer_capacity: int = None,
    R: int = 5,
    E: int = 1,
    lr: float = 1e-2,
    batch_size: int = 32,
    lambda_ewc: float = 100.0,
    adapt_all_clients_on_global_drift: bool = True,
    clone_models: bool = True,
):
    print("\nRunning EWC Continual Learning...")

    models, drift_counts, exceed_count, stream_rmses, client_predictions = _initialize_stream_state(
        task_lagged_base_test, personalized_predictors, device, clone_models
    )

    if recent_buffer_capacity is None:
        recent_buffer_capacity = persistence_M * pred_len

    recent_buffers = {c: RecentStreamBuffer(recent_buffer_capacity) for c in task_lagged_base_test.keys()}
    theta = dict(theta_override) if theta_override is not None else {}

    min_len = min(len(v["y"]) for v in task_lagged_base_test.values())
    max_chunks = max(0, min_len // pred_len)
    start_chunk = warmup_windows
    global_drift_events = 0
    drift_history = []

    if start_chunk >= max_chunks:
        return _finalize_result(models, recent_buffers, drift_counts, 0, [], stream_rmses, client_predictions)

    for chunk_id in range(start_chunk, max_chunks):
        t = chunk_id * pred_len

        def update_recent(client, X_full, Y_full, t0, p):
            recent_buffers[client].add_batch(_make_current_chunk_pairs(X_full, Y_full, t0, p))

        drifting_clients = _stream_predict_detect_update_buffers(
            models=models,
            task_lagged_base_test=task_lagged_base_test,
            device=device,
            pred_len=pred_len,
            t=t,
            theta=theta,
            rmse_threshold_fallback=rmse_threshold_fallback,
            persistence_M=persistence_M,
            drift_counts=drift_counts,
            exceed_count=exceed_count,
            stream_rmses=stream_rmses,
            client_predictions=client_predictions,
            buffer_update_fn=update_recent,
        )

        if len(drifting_clients) == 0:
            continue

        global_drift_events += 1
        drift_history.append({"chunk_id": chunk_id, "t": t, "drifting_clients": drifting_clients})
        participating_clients = list(task_lagged_base_test.keys()) if adapt_all_clients_on_global_drift else drifting_clients

        for _round in range(R):
            local_states, local_weights = [], []

            for client in participating_clients:
                pairs = recent_buffers[client].get_all_pairs()
                if len(pairs) == 0:
                    continue

                local_model = deepcopy(models[client]).to(device).train()
                optimizer = torch.optim.Adam(local_model.parameters(), lr=lr)
                fisher = ewc_fishers_by_client[client]
                # Fisher and reference parameters are computed from base data.
                ref_params = ewc_ref_params_by_client[client]

                for _ in range(E):
                    for Xb, Yb in iterate_subset_batches(pairs, batch_size=batch_size, device=device, shuffle=False):
                        optimizer.zero_grad(set_to_none=True)
                        loss_current = F.mse_loss(local_model(Xb), Yb)
                        loss_ewc = compute_ewc_penalty(local_model, fisher, ref_params)
                        loss = loss_current + lambda_ewc * loss_ewc
                        # EWC trades off stream fitting against parameter preservation.
                        loss.backward()
                        optimizer.step()
                     

                local_states.append({k: v.detach().cpu().clone() for k, v in local_model.state_dict().items()})
                local_weights.append(len(pairs))

            if len(local_states) > 0:
                avg_state = _weighted_fedavg_state_dicts(local_states, local_weights)
                for client in models:
                    models[client].load_state_dict(avg_state)
                    models[client].to(device).eval()

        for client in exceed_count:
            exceed_count[client] = 0

    return _finalize_result(models, recent_buffers, drift_counts, global_drift_events, drift_history, stream_rmses, client_predictions)


# ============================================================
# 16. Method 5: LwF
# ============================================================
# Uses a frozen teacher snapshot to preserve previous predictions during
# stream adaptation.

def LwFContinualLearning(
    *,
    task_lagged_base_test: dict,
    personalized_predictors: dict,
    device: str = "cpu",
    pred_len: int = 6,
    theta_override: dict = None,
    warmup_windows: int = 5,
    rmse_threshold_fallback: float = 0.22,
    persistence_M: int = 2,
    recent_buffer_capacity: int = None,
    R: int = 5,
    E: int = 1,
    lr: float = 1e-2,
    batch_size: int = 32,
    lambda_lwf: float = 1.0,
    adapt_all_clients_on_global_drift: bool = True,
    clone_models: bool = True,
):
    print("\nRunning LwF Continual Learning...")

    models, drift_counts, exceed_count, stream_rmses, client_predictions = _initialize_stream_state(
        task_lagged_base_test, personalized_predictors, device, clone_models
    )

    if recent_buffer_capacity is None:
        recent_buffer_capacity = persistence_M * pred_len

    recent_buffers = {c: RecentStreamBuffer(recent_buffer_capacity) for c in task_lagged_base_test.keys()}
    theta = dict(theta_override) if theta_override is not None else {}

    min_len = min(len(v["y"]) for v in task_lagged_base_test.values())
    max_chunks = max(0, min_len // pred_len)
    start_chunk = warmup_windows
    global_drift_events = 0
    drift_history = []

    if start_chunk >= max_chunks:
        return _finalize_result(models, recent_buffers, drift_counts, 0, [], stream_rmses, client_predictions)

    for chunk_id in range(start_chunk, max_chunks):
        t = chunk_id * pred_len

        def update_recent(client, X_full, Y_full, t0, p):
            recent_buffers[client].add_batch(_make_current_chunk_pairs(X_full, Y_full, t0, p))

        drifting_clients = _stream_predict_detect_update_buffers(
            models=models,
            task_lagged_base_test=task_lagged_base_test,
            device=device,
            pred_len=pred_len,
            t=t,
            theta=theta,
            rmse_threshold_fallback=rmse_threshold_fallback,
            persistence_M=persistence_M,
            drift_counts=drift_counts,
            exceed_count=exceed_count,
            stream_rmses=stream_rmses,
            client_predictions=client_predictions,
            buffer_update_fn=update_recent,
        )

        if len(drifting_clients) == 0:
            continue

        global_drift_events += 1
        drift_history.append({"chunk_id": chunk_id, "t": t, "drifting_clients": drifting_clients})
        participating_clients = list(task_lagged_base_test.keys()) if adapt_all_clients_on_global_drift else drifting_clients

        teachers = {c: deepcopy(models[c]).to(device).eval() for c in participating_clients if c in models}
        # Freeze the pre-adaptation model as the teacher for this drift event.
        for teacher in teachers.values():
            for p in teacher.parameters():
                p.requires_grad = False

        for _round in range(R):
            local_states, local_weights = [], []

            for client in participating_clients:
                pairs = recent_buffers[client].get_all_pairs()
                if len(pairs) == 0 or client not in teachers:
                    continue

                local_model = deepcopy(models[client]).to(device).train()
                teacher = teachers[client]
                optimizer = torch.optim.Adam(local_model.parameters(), lr=lr)

                for _ in range(E):
                    for Xb, Yb in iterate_subset_batches(pairs, batch_size=batch_size, device=device, shuffle=False):
                        optimizer.zero_grad(set_to_none=True)
                        pred_student = local_model(Xb)
                        with torch.no_grad():
                            pred_teacher = teacher(Xb)
                        loss_current = F.mse_loss(pred_student, Yb)
                        loss_kd = compute_lwf_kd_loss(pred_student, pred_teacher)
                        loss = loss_current + lambda_lwf * loss_kd
                        # LwF combines supervised stream loss with distillation loss.
                        loss.backward()
                        optimizer.step()

                local_states.append({k: v.detach().cpu().clone() for k, v in local_model.state_dict().items()})
                local_weights.append(len(pairs))

            if len(local_states) > 0:
                avg_state = _weighted_fedavg_state_dicts(local_states, local_weights)
                for client in models:
                    models[client].load_state_dict(avg_state)
                    models[client].to(device).eval()

        for client in exceed_count:
            exceed_count[client] = 0

    return _finalize_result(models, recent_buffers, drift_counts, global_drift_events, drift_history, stream_rmses, client_predictions)


# ============================================================
# 17. Method 6: Synaptic Intelligence
# ============================================================
# Uses SI importance weights accumulated during base training to protect
# important parameters during later adaptation.

def SIContinualLearning(
    *,
    task_lagged_base_test: dict,
    personalized_predictors: dict,
    si_omegas_by_client: dict,
    si_ref_params_by_client: dict,
    device: str = "cpu",
    pred_len: int = 6,
    theta_override: dict = None,
    warmup_windows: int = 5,
    rmse_threshold_fallback: float = 0.22,
    persistence_M: int = 2,
    recent_buffer_capacity: int = None,
    R: int = 5,
    E: int = 1,
    lr: float = 1e-4,
    batch_size: int = 32,
    lambda_si: float = 1.0,
    adapt_all_clients_on_global_drift: bool = True,
    clone_models: bool = True,
):
    print("\nRunning SI Continual Learning...")

    models, drift_counts, exceed_count, stream_rmses, client_predictions = _initialize_stream_state(
        task_lagged_base_test, personalized_predictors, device, clone_models
    )

    if recent_buffer_capacity is None:
        recent_buffer_capacity = persistence_M * pred_len

    recent_buffers = {c: RecentStreamBuffer(recent_buffer_capacity) for c in task_lagged_base_test.keys()}
    theta = dict(theta_override) if theta_override is not None else {}

    min_len = min(len(v["y"]) for v in task_lagged_base_test.values())
    max_chunks = max(0, min_len // pred_len)
    start_chunk = warmup_windows
    global_drift_events = 0
    drift_history = []

    if start_chunk >= max_chunks:
        return _finalize_result(models, recent_buffers, drift_counts, 0, [], stream_rmses, client_predictions)

    for chunk_id in range(start_chunk, max_chunks):
        t = chunk_id * pred_len

        def update_recent(client, X_full, Y_full, t0, p):
            recent_buffers[client].add_batch(_make_current_chunk_pairs(X_full, Y_full, t0, p))

        drifting_clients = _stream_predict_detect_update_buffers(
            models=models,
            task_lagged_base_test=task_lagged_base_test,
            device=device,
            pred_len=pred_len,
            t=t,
            theta=theta,
            rmse_threshold_fallback=rmse_threshold_fallback,
            persistence_M=persistence_M,
            drift_counts=drift_counts,
            exceed_count=exceed_count,
            stream_rmses=stream_rmses,
            client_predictions=client_predictions,
            buffer_update_fn=update_recent,
        )

        if len(drifting_clients) == 0:
            continue

        global_drift_events += 1
        drift_history.append({"chunk_id": chunk_id, "t": t, "drifting_clients": drifting_clients})
        participating_clients = list(task_lagged_base_test.keys()) if adapt_all_clients_on_global_drift else drifting_clients

        for _round in range(R):
            local_states, local_weights = [], []

            for client in participating_clients:
                pairs = recent_buffers[client].get_all_pairs()
                if len(pairs) == 0:
                    continue

                local_model = deepcopy(models[client]).to(device).train()
                optimizer = torch.optim.Adam(local_model.parameters(), lr=lr)
                omega = si_omegas_by_client[client]
                # Omega contains SI parameter-importance estimates from base training.
                ref_params = si_ref_params_by_client[client]

                for _ in range(E):
                    for Xb, Yb in iterate_subset_batches(pairs, batch_size=batch_size, device=device, shuffle=False):
                        optimizer.zero_grad(set_to_none=True)
                        loss_current = F.mse_loss(local_model(Xb), Yb)
                        loss_si = compute_si_penalty(local_model, omega, ref_params)
                        loss = loss_current + lambda_si * loss_si
                        # SI regularization discourages changes to important parameters.
                        loss.backward()
                        optimizer.step()
                

                local_states.append({k: v.detach().cpu().clone() for k, v in local_model.state_dict().items()})
                local_weights.append(len(pairs))

            if len(local_states) > 0:
                avg_state = _weighted_fedavg_state_dicts(local_states, local_weights)
                for client in models:
                    models[client].load_state_dict(avg_state)
                    models[client].to(device).eval()

        for client in exceed_count:
            exceed_count[client] = 0

    return _finalize_result(models, recent_buffers, drift_counts, global_drift_events, drift_history, stream_rmses, client_predictions)


print("✅ methods and helper definitions loaded successfully.")

✅ methods and helper definitions loaded successfully.


In [ ]:
# ============================================================
# EXPERIMENT RUNNER NOTEBOOK
# This notebook coordinates the full experimental workflow: data loading,
# preprocessing, base federated training, continual-learning evaluation,
# aggregation of metrics, and saving of paper-ready result tables.
# ============================================================

import os
import time
import random
import copy
import warnings
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.cluster import KMeans

# Suppress non-critical warnings to keep long notebook runs readable.
warnings.filterwarnings("ignore")


# ============================================================
# 0. GLOBAL CONFIGURATION
# All experimental choices are centralized here to make the notebook reproducible.
# For quick debugging, reduce the number of seeds, targets, horizons, or base rounds.
# ============================================================

# Central experiment dictionary: changing this block controls the full grid.
CONFIG = {
    # Dataset
    "dataset_dir": Path("../dataset/AirQuality"),
    "dataset_names": [
        "Aotizhongxin.csv", "Changping.csv", "Dingling.csv",
        "Dongsi.csv", "Guanyuan.csv", "Gucheng.csv",
        "Huairou.csv", "Nongzhanguan.csv", "Shunyi.csv",
        "Tiantan.csv", "Wanliu.csv", "Wanshouxigong.csv"
    ],

    # Paper dates
    # Adjust here if you want a shorter debugging run.
    "base_start": "2013-03-01",
    "base_end": "2014-03-01 23:00:00",

    "stream_start": "2014-03-02",
    "stream_end": "2017-03-03 23:00:00",

    # Forecasting setup      ,
    "targets": [    "TEMP","PM2.5","WSPM"], 
    "horizons": [ 6, 12,  24],
    "seeds": [42, 43, 44],

    
    # Lag length
    "n_lags": 12,

    # Dataloader
    "batch_size": 32,

    # Model
    "hidden_dim": 64,

    # Base FL training
    "base_rounds": 100,       
    "base_local_epochs": 1,
    "base_lr": 1e-4,

    # Drift detection
    "warmup_windows": 1,
    "k_mad": 1,
    "rmse_threshold_fallback": 0.22,
    "persistence_M": 2,

    # Replay buffers
    "buffer_capacity": 20000,
    "sample_fraction": 0.10,

    # CL adaptation
    "R": 5,
    "E": 1,
    "cl_lr": 1e-3,

    # Replay parameters
    "replay_fraction": 0.30,
    "replay_min_samples": 32,
    "replay_max_samples": 300,
    "lambda_current": 0.5,
    "lambda_replay": 2,
    "p_offline": 0.8,
    "online_capacity": 300,

    # Regularization/distillation
    "lambda_ewc": 1e4,

    # Target-specific regularization and distillation strengths.
    # These values are selected according to the forecasting target.
    "target_regularization": {
        "PM2.5": {
            "lambda_lwf": 1,
            "lambda_si": 0.5,
        },
        "WSPM": {
            "lambda_lwf": 10,
            "lambda_si": 1,
        },
        "TEMP": {
            "lambda_lwf": 0.1,
            "lambda_si": 0.01,
        },
    },

    # Output
    "results_dir": "results",

    # Methods  
    "methods": [
        "Static",
        "Naive",
        "NormalReplay",
        "TwoTierReplay",
        "EWC",
        "LwF",
        "SI"
    ],
}


# Create the output directory before saving intermediate and final results.
os.makedirs(CONFIG["results_dir"], exist_ok=True)

# Use GPU when available; otherwise fall back to CPU for portability.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Using device: {device}")


# ============================================================
# 1. REPRODUCIBILITY
# Seeding is called at the beginning of each experimental context.
# ============================================================

def set_all_seeds(seed=42):
    # Set all relevant random seeds so repeated runs are as reproducible as possible.
    # The deterministic backend flags reduce nondeterminism from CUDA/cuDNN operations.
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


# ============================================================
# 2. DATA PREPROCESSING
# This block transforms raw Beijing air-quality station files into normalized
# lagged forecasting samples for federated learning.
# ============================================================

# Final feature list used by all clients after preprocessing.
FEATURES = [
    "No", "year",
    "PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
    "TEMP", "PRES", "DEWP", "RAIN", "WSPM",
    "wd_sin", "wd_cos",
    "hour_sin", "hour_cos",
    "month_sin", "month_cos",
    "day_sin", "day_cos"
]

# Mapping from categorical wind direction to compass angle in degrees.
WD_TO_ANGLE = {
    "N": 0, "NNE": 22.5, "NE": 45, "ENE": 67.5,
    "E": 90, "ESE": 112.5, "SE": 135, "SSE": 157.5,
    "S": 180, "SSW": 202.5, "SW": 225, "WSW": 247.5,
    "W": 270, "WNW": 292.5, "NW": 315, "NNW": 337.5
}


def smart_imputation(df):
    # Work on a copy to avoid modifying the original client DataFrame in place.
    # Numerical features are interpolated over time; wind direction is forward/backward filled.
    df = df.copy()
    cont_features = df.select_dtypes(include=[np.number]).columns
    df[cont_features] = df[cont_features].interpolate(
        method="time",
        limit_direction="both"
    )

    if "wd" in df.columns:
        df["wd"] = df["wd"].ffill().bfill()

    # Drop any rows still containing missing values after interpolation/filling.
    df = df.dropna()
    return df


def encode_wind_direction(df):
    # Encode categorical wind direction as sine/cosine components to preserve circular geometry.
    # For example, N and NNW remain close in the transformed representation.
    df = df.copy()

    if "wd" in df.columns:
        angles = df["wd"].map(WD_TO_ANGLE).fillna(0.0)
        radians = np.deg2rad(angles)
        df["wd_sin"] = np.sin(radians)
        df["wd_cos"] = np.cos(radians)
    else:
        df["wd_sin"] = 0.0
        df["wd_cos"] = 0.0

    return df


def add_cyclical_time(df):
    # Encode calendar variables cyclically so boundaries such as 23:00 -> 00:00 are continuous.
    df = df.copy()

    if all(c in df.columns for c in ["hour", "month", "day"]):
        df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
        df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

        df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
        df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

        df["day_sin"] = np.sin(2 * np.pi * df["day"] / 31)
        df["day_cos"] = np.cos(2 * np.pi * df["day"] / 31)

    return df


def load_and_preprocess_clients(config):
    # Load each monitoring station as one federated client.
    # Each CSV is indexed by hourly timestamp and processed using the same feature pipeline.
    dataset_dir = config["dataset_dir"]
    dataset_names = config["dataset_names"]

    client_names = [name.replace(".csv", "") for name in dataset_names]
    client_dfs = {}

    for name in dataset_names:
        df = pd.read_csv(dataset_dir / name)
        df["time"] = pd.to_datetime(df[["year", "month", "day", "hour"]])
        df = df.set_index("time").sort_index()

        df = smart_imputation(df)
        df = encode_wind_direction(df)
        df = add_cyclical_time(df)

        # Ensure all clients share the same feature schema.
        missing = [c for c in FEATURES if c not in df.columns]
        for c in missing:
            df[c] = 0.0

        client_dfs[name.replace(".csv", "")] = df[FEATURES].copy()

    return client_dfs, client_names


def compute_global_bounds(client_dfs):
    # Compute robust feature bounds across all clients using 1st and 99th percentiles.
    # This helper is retained for completeness; the experiment uses the base-only version below.
    feature_percentiles = defaultdict(list)

    for _, df in client_dfs.items():
        for col in FEATURES:
            if pd.api.types.is_numeric_dtype(df[col]):
                vals = df[col].dropna().values
                if vals.size > 0:
                    lower = np.percentile(vals, 1)
                    upper = np.percentile(vals, 99)
                    feature_percentiles[col].append((lower, upper))

    global_bounds = {
        col: (min(p[0] for p in vals), max(p[1] for p in vals))
        for col, vals in feature_percentiles.items()
    }

    return global_bounds


def compute_global_bounds_base_only(client_dfs, config):
    # Compute robust normalization bounds using only the offline/base period.
    # This prevents information leakage from the future continual-learning stream.
    """
    Compute robust global min-max bounds using only the offline/base training period.

    This avoids using future evaluation/streaming data for normalization.
    """
    feature_percentiles = defaultdict(list)

    for _, df in client_dfs.items():
        base_df = df.loc[config["base_start"]:config["base_end"]]

        for col in FEATURES:
            if col in base_df.columns and pd.api.types.is_numeric_dtype(base_df[col]):
                vals = base_df[col].dropna().values

                if vals.size > 0:
                    lower = np.percentile(vals, 1)
                    upper = np.percentile(vals, 99)
                    feature_percentiles[col].append((lower, upper))

    global_bounds = {}

    for col, vals in feature_percentiles.items():
        if len(vals) > 0:
            global_bounds[col] = (
                min(p[0] for p in vals),
                max(p[1] for p in vals)
            )

    return global_bounds


def normalize_df_globally(df, global_bounds, required_columns, skip_columns=None):
    # Apply min-max normalization using precomputed global bounds.
    # Features listed in skip_columns are kept unchanged.
    df = df.copy()
    df = df[required_columns]

    if skip_columns is None:
        skip_columns = []

    for col in required_columns:
        if col in skip_columns:
            continue

        if col in global_bounds and pd.api.types.is_numeric_dtype(df[col]):
            mn, mx = global_bounds[col]

            if mx != mn:
                df[col] = (df[col] - mn) / (mx - mn)
                df[col] = df[col].clip(0.0, 1.0)
            else:
                df[col] = 0.0

    return df


def split_data_by_custom_range(df, config):
    # Split each client into an offline base period and a later streaming period.
    base_df = df.loc[config["base_start"]:config["base_end"]]
    stream_df = df.loc[config["stream_start"]:config["stream_end"]]

    return base_df, stream_df


def create_lagged_samples(
    df,
    n_lags,
    pred_len,
    target_col,
    include_target_in_input=True
):
    # Convert a normalized time series into supervised forecasting samples.
    # Each input contains n_lags historical rows; each target contains pred_len future values.
    if include_target_in_input:
        feature_cols = df.columns.tolist()
    else:
        feature_cols = [c for c in df.columns if c != target_col]

    X, y = [], []

    feature_vals = df[feature_cols].astype(np.float64).values
    target_vals = df[[target_col]].astype(np.float64).values

    for i in range(n_lags, len(df) - pred_len + 1):
        lagged = feature_vals[i - n_lags:i]
        tgtseq = target_vals[i:i + pred_len].flatten()

        if np.isnan(lagged).any() or np.isnan(tgtseq).any():
            continue

        X.append(lagged)
        y.append(tgtseq)

    return np.asarray(X), np.asarray(y)


class TimeSeriesDataset(Dataset):
    # Minimal PyTorch Dataset wrapper for lagged time-series arrays.
    def __init__(self, X, y):
        # Store tensors once so DataLoader iteration does not repeatedly convert NumPy arrays.
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def prepare_lagged_data_and_loaders(
    client_dfs,
    global_bounds,
    target_col,
    pred_len,
    config
):
    # Build lagged arrays and DataLoaders for one target/horizon pair.
    # The base period is used for initial FL training and retention evaluation.
    # The stream period is used for task-free continual-learning evaluation.
    n_lags = config["n_lags"]
    batch_size = config["batch_size"]

    base_lagged = {}

    # Keep "base_test" for compatibility with your existing CL methods.
    # Scientifically, this is the CL stream.
    task_lagged = {
        "base_test": {},
        "cl_stream": {}
    }

    for client, df in client_dfs.items():
        base_df, stream_df = split_data_by_custom_range(df, config)

        base_df_norm = normalize_df_globally(
            base_df, global_bounds, FEATURES, skip_columns=["No"]
        )

        stream_df_norm = normalize_df_globally(
            stream_df, global_bounds, FEATURES, skip_columns=["No"]
        )

        X_base, y_base = create_lagged_samples(
            base_df_norm, n_lags, pred_len, target_col
        )

        X_stream, y_stream = create_lagged_samples(
            stream_df_norm, n_lags, pred_len, target_col
        )

        base_lagged[client] = {"X": X_base, "y": y_base}

        # Existing code expects task_lagged["base_test"], so keep it.
        task_lagged["base_test"][client] = {"X": X_stream, "y": y_stream}

        # Cleaner scientific alias.
        task_lagged["cl_stream"][client] = {"X": X_stream, "y": y_stream}

    # Organize loaders by their experimental role.
    client_dataloaders = {
        "base_train": {},
        "base_retain": {},
        "base_test": {},
    }

    for client in base_lagged:
        Xb, yb = base_lagged[client]["X"], base_lagged[client]["y"]
        Xs, ys = task_lagged["base_test"][client]["X"], task_lagged["base_test"][client]["y"]

        # Base training loader used during the offline FedAvg phase.
        client_dataloaders["base_train"][client] = DataLoader(
            TimeSeriesDataset(Xb, yb),
            batch_size=batch_size,
            shuffle=False
        )

        # This is intentionally the same base-period dataset.
        # It is used to measure retention of what the base model learned.
        # Retention loader used to measure forgetting after CL adaptation.
        client_dataloaders["base_retain"][client] = DataLoader(
            TimeSeriesDataset(Xb, yb),
            batch_size=batch_size,
            shuffle=False
        )

        # Stream loader kept for compatibility with the earlier naming convention.
        client_dataloaders["base_test"][client] = DataLoader(
            TimeSeriesDataset(Xs, ys),
            batch_size=batch_size,
            shuffle=False
        )

    return base_lagged, task_lagged, client_dataloaders



# ============================================================
# 3. NORMAL REPLAY BUFFER FOR BASELINE
# This buffer is used only by the NormalReplay baseline, where all replay
# samples share one FIFO memory.
# ============================================================

class NormalReplayBuffer:
    # Standard single-memory FIFO replay buffer for the NormalReplay baseline.
    def __init__(self, capacity):
        # deque(maxlen=capacity) automatically removes the oldest sample when full.
        self.capacity = int(capacity)
        self.buffer = deque(maxlen=self.capacity)

    def add(self, x, y):
        if self.capacity <= 0:
            return
        self.buffer.append((x.detach().cpu(), y.detach().cpu()))

    def add_batch(self, pairs):
        for x, y in pairs:
            self.add(x, y)

    def sample_subset(self, subset_size):
        total = len(self.buffer)

        if total == 0:
            return []

        subset_size = min(int(subset_size), total)

        if subset_size <= 0:
            return []

        idx = np.random.choice(total, size=subset_size, replace=False)
        buffer_list = list(self.buffer)

        return [buffer_list[i] for i in idx]

    def __len__(self):
        return len(self.buffer)

    def get_all_pairs(self):
        return list(self.buffer)


def build_normal_replay_buffers_from_base(initial_buffers, capacity):
    # Initialize NormalReplay using the same base representative samples as TwoTierReplay.
    # This keeps the comparison fair at the beginning of the stream.
    normal_buffers = {}

    for client, buf in initial_buffers.items():
        normal_buf = NormalReplayBuffer(capacity=capacity)

        if hasattr(buf, "offline_fixed"):
            base_pairs = list(buf.offline_fixed)
        elif hasattr(buf, "buffer"):
            base_pairs = list(buf.buffer)
        else:
            base_pairs = list(buf)

        normal_buf.add_batch(base_pairs)
        normal_buffers[client] = normal_buf

    return normal_buffers


# ============================================================
# 4. EVALUATION HELPERS
# These helpers compute RMSE, forgetting, and stability-plasticity summaries.
# ============================================================

@torch.no_grad()
def evaluate_models_on_dataloaders(models_by_client, dataloaders_by_client, device="cpu"):
    # Evaluate each client model on the corresponding client DataLoader and report RMSE.
    per_client_rmse = {}

    for client, loader in dataloaders_by_client.items():
        if client not in models_by_client:
            continue

        model = models_by_client[client].to(device).eval()

        se_sum, n = 0.0, 0

        for X, y in loader:
            X, y = X.to(device), y.to(device)
            y_hat = model(X)

            se_sum += torch.sum((y_hat - y) ** 2).item()
            n += y.numel()

        rmse = float(np.sqrt(se_sum / max(1, n)))
        per_client_rmse[client] = rmse

    valid = [v for v in per_client_rmse.values() if np.isfinite(v)]
    avg_rmse = float(np.mean(valid)) if valid else float("nan")

    return per_client_rmse, avg_rmse


@torch.no_grad()
def evaluate_static_stream_rmse(
    models_by_client,
    task_lagged_base_test,
    device,
    pred_len,
    warmup_windows,
):
    # Evaluate the static, non-adaptive baseline on the same stream chunks.
    # This provides a reference for the value of continual adaptation.
    stream_rmses = {}

    min_len = min(len(v["y"]) for v in task_lagged_base_test.values())
    max_chunks = max(0, min_len // pred_len)
    start_chunk = warmup_windows

    for client, data in task_lagged_base_test.items():
        model = models_by_client[client].to(device).eval()
        X_full, y_full = data["X"], data["y"]

        rmses = []

        for chunk_id in range(start_chunk, max_chunks):
            t = chunk_id * pred_len

            if t >= len(y_full):
                break

            x = torch.tensor(
                X_full[t],
                dtype=torch.float32,
                device=device
            ).unsqueeze(0)

            y_true = y_full[t].astype(np.float32).ravel()
            y_pred = model(x).cpu().numpy().ravel()

            if y_pred.shape != y_true.shape:
                continue

            rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
            rmses.append(rmse)

        stream_rmses[client] = rmses

    return {
        "stream_rmses": stream_rmses,
        "final_models": models_by_client,
        "drift_counts": {c: 0 for c in task_lagged_base_test.keys()},
        "total_drifts": 0,
        "avg_drifts_per_client": 0.0,
        "global_drift_events": 0,
        "client_predictions": {}
    }


def mean_stream_rmse(stream_rmses):
    # Average the stream RMSE traces across clients.
    vals = []

    for _, rmses in stream_rmses.items():
        if isinstance(rmses, list) and len(rmses) > 0:
            vals.append(float(np.mean(rmses)))
        elif isinstance(rmses, (int, float, np.float32, np.float64)):
            vals.append(float(rmses))

    return float(np.mean(vals)) if vals else float("nan")


def compute_base_reference(ctx):
    # Measure the initial offline FedAvg model on the base-period retain set.
    # This reference is later used to quantify forgetting after CL adaptation.
    """
    Reference performance of the offline base model on the base-period data.

    This is the baseline for measuring forgetting/stability.
    """

    base_rmse_base_model, avg_base_rmse_base_model = evaluate_models_on_dataloaders(
        models_by_client=ctx["base_predictors"],
        dataloaders_by_client=ctx["client_dataloaders"]["base_retain"],
        device=ctx["device"]
    )

    return {
        "base_rmse_base_model": base_rmse_base_model,
        "avg_base_rmse_base_model": avg_base_rmse_base_model,
    }


def evaluate_method_result(method_name, result, ctx, cpu_time, base_reference):
    # Convert one method's raw result dictionary into summary and per-client rows.
    # The main score combines stream plasticity and signed base-period forgetting.
    """
    Evaluate one method under the intended protocol:

    Plasticity:
        stream_rmse = online CL-stream RMSE

    Stability / Forgetting:
        af_base_retain = RMSE(final CL model on base period)
                         - RMSE(offline base model on base period)

    Negative af_base_retain means improved base retention / positive backward transfer.
    """

    final_models = result["final_models"]

    base_rmse_final_model, avg_base_rmse_final_model = evaluate_models_on_dataloaders(
        models_by_client=final_models,
        dataloaders_by_client=ctx["client_dataloaders"]["base_retain"],
        device=ctx["device"]
    )

    stream_rmse = mean_stream_rmse(result.get("stream_rmses", {}))

    af_base_retain = (
        avg_base_rmse_final_model
        - base_reference["avg_base_rmse_base_model"]
    )

    row = {
        "target": ctx["target_col"],
        "horizon": ctx["pred_len"],
        "seed": ctx["seed"],
        "method": method_name,

        # Plasticity / adaptation
        "stream_rmse": stream_rmse,

        # Stability / forgetting
        "base_model_base_rmse": base_reference["avg_base_rmse_base_model"],
        "final_model_base_rmse": avg_base_rmse_final_model,
        "af_base_retain": af_base_retain,

        # Drift / cost
        "total_drifts": result.get("total_drifts", 0),
        "avg_drifts_client": result.get("avg_drifts_per_client", 0.0),
        "global_drift_events": result.get("global_drift_events", np.nan),
        "cpu_sec": cpu_time,
    }

    per_client_rows = []

    for client in ctx["client_names"]:
        base_ref_client = base_reference["base_rmse_base_model"].get(client, np.nan)
        final_base_client = base_rmse_final_model.get(client, np.nan)

        per_client_rows.append({
            "target": ctx["target_col"],
            "horizon": ctx["pred_len"],
            "seed": ctx["seed"],
            "method": method_name,
            "client": client,

            "base_model_base_rmse": base_ref_client,
            "final_model_base_rmse": final_base_client,
            "af_base_retain": final_base_client - base_ref_client,

            "drift_count": result.get("drift_counts", {}).get(client, 0),
        })

    return row, per_client_rows



from copy import deepcopy
import time
import pandas as pd
import numpy as np
import torch

def make_global_model_dict(base_predictor, client_names, device):
    # Return client-indexed copies of the final global model for evaluation convenience.
    """
    Create one copy of the final global FedAvg model for each client.
    This guarantees evaluation is based on the shared global base model,
    not last-round local client models.
    """
    return {
        client: deepcopy(base_predictor).to(device).eval()
        for client in client_names
    }

# ============================================================
# 5. BUFFER AND THRESHOLD HELPERS
# These functions build replay memories and prepare warm-up samples.
# ============================================================

def build_two_tier_buffers(base_predictors, client_dataloaders, config, device):
    # Build a TwoTierReplayBuffer for each client using KMeans-selected base samples.
    client_replay_buffers = {}

    for client, loader in client_dataloaders["base_train"].items():
        base_buf = init_base_buffer(
            predictor=base_predictors[client],
            loader=loader,
            device=device,
            buffer_capacity=config["buffer_capacity"],
            sample_fraction=config["sample_fraction"]
        )

        client_replay_buffers[client] = upgrade_to_two_tier(
            base_buf,
            total_capacity=config["buffer_capacity"],
            online_capacity=config["online_capacity"]
        )

    return client_replay_buffers


def reset_online_zone(buffers):
    # Clear only the online FIFO tier while preserving the fixed offline base memory.
    for _, buf in buffers.items():
        if hasattr(buf, "online_fifo"):
            buf.online_fifo.clear()

            if hasattr(buf, "_bump_ver"):
                buf._bump_ver()


def prefill_online_with_calibration_windows(task_lagged_base_test, buffers, warmup_windows, pred_len):
    # Insert warm-up stream windows into online buffers before scoring begins.
    # These windows are skipped during evaluation and therefore serve only as calibration context.
    if warmup_windows <= 0:
        return

    T = warmup_windows * pred_len

    for client, data in task_lagged_base_test.items():
        buf = buffers.get(client, None)

        if buf is None:
            continue

        upper = min(T, len(data["X"]), len(data["y"]))

        pending = []
        for idx in range(0, upper):
            x_i = torch.tensor(data["X"][idx], dtype=torch.float32)
            y_i = torch.tensor(data["y"][idx], dtype=torch.float32)
            pending.append((x_i, y_i))

        if hasattr(buf, "add_batch_online"):
            buf.add_batch_online(pending)
        elif hasattr(buf, "add"):
            for x_i, y_i in pending:
                buf.add(x_i, y_i)


# ============================================================
# 6. BUILD CONTEXT FOR ONE TARGET/HORIZON/SEED
# A context stores every object required to run all methods under one condition.
# ============================================================

def build_context_for_experiment(
    target_col,
    pred_len,
    seed,
    client_dfs,
    client_names,
    global_bounds,
    config,
):
    # Prepare all objects needed for one experimental condition:
    # data loaders, base FedAvg model, drift thresholds, replay buffers, and regularizers.
    set_all_seeds(seed)

    print("\n" + "#" * 100)
    print(f"Preparing context: target={target_col}, horizon={pred_len}, seed={seed}")
    print("#" * 100)

    base_lagged, task_lagged, client_dataloaders = prepare_lagged_data_and_loaders(
        client_dfs=client_dfs,
        global_bounds=global_bounds,
        target_col=target_col,
        pred_len=pred_len,
        config=config
    )
    
    

    sample_client = list(base_lagged.keys())[0]
    input_dim = base_lagged[sample_client]["X"].shape[2]
    output_dim = pred_len

    base_predictor, base_predictors, base_history, si_ref_params_by_client, si_omegas_by_client = train_federated_base_model(
        client_dataloaders=client_dataloaders,
        input_dim=input_dim,
        output_dim=output_dim,
        num_rounds=config["base_rounds"],
        local_epochs=config["base_local_epochs"],
        hidden_dim=config["hidden_dim"],
        lr=config["base_lr"],
        device=device,
        eval_every=1,
        si_eps=1e-4
    )

    theta_shared, baseline_shared = calibrate_theta_mad_from_base(
        base_lagged=base_lagged,
        models_by_client=base_predictors,
        device=device,
        k=config["k_mad"],
        min_mad=1e-6,
        verbose=False
    )

    client_replay_buffers = build_two_tier_buffers(
        base_predictors=base_predictors,
        client_dataloaders=client_dataloaders,
        config=config,
        device=device
    )

    reset_online_zone(client_replay_buffers)

    prefill_online_with_calibration_windows(
        task_lagged_base_test=task_lagged["base_test"],
        buffers=client_replay_buffers,
        warmup_windows=config["warmup_windows"],
        pred_len=pred_len
    )

    normal_replay_buffers = build_normal_replay_buffers_from_base(
        initial_buffers=client_replay_buffers,
        capacity=config["buffer_capacity"]
    )

    ewc_fishers_by_client = {}
    ewc_ref_params_by_client = {}

    for client in client_names:
        model = base_predictors[client].to(device).eval()

        fisher = compute_ewc_fisher(
            model=model,
            dataloader=client_dataloaders["base_train"][client],
            device=device,
            num_batches=10
        )

        ref_params = {
            n: p.detach().clone().to(device)
            for n, p in model.named_parameters()
        }

        ewc_fishers_by_client[client] = fisher
        ewc_ref_params_by_client[client] = ref_params

    ctx = {
        "target_col": target_col,
        "pred_len": pred_len,
        "seed": seed,
        "device": device,
        "client_names": client_names,
        "base_lagged": base_lagged,
        "task_lagged": task_lagged,
        "client_dataloaders": client_dataloaders,
        "base_predictor": base_predictor,
        "base_predictors": base_predictors,
        "theta_shared": theta_shared,
        "baseline_shared": baseline_shared,

        "client_replay_buffers": client_replay_buffers,
        "normal_replay_buffers": normal_replay_buffers,

        "ewc_fishers_by_client": ewc_fishers_by_client,
        "ewc_ref_params_by_client": ewc_ref_params_by_client,

        "si_omegas_by_client": si_omegas_by_client,
        "si_ref_params_by_client": si_ref_params_by_client,

        "warmup_windows": config["warmup_windows"],
    }

    return ctx



def clone_global_model_for_clients(global_model, client_names, device):
    # Clone a shared global model into one model dictionary keyed by client name.
    return {
        client: deepcopy(global_model).to(device).eval()
        for client in client_names
    }


def model_l2_distance(model_a, model_b):
    # Diagnostic helper: compute absolute and relative L2 distance between two models.
    sq_sum = 0.0
    norm_sum = 0.0

    with torch.no_grad():
        for p_a, p_b in zip(model_a.parameters(), model_b.parameters()):
            diff = p_a.detach().cpu() - p_b.detach().cpu()
            sq_sum += torch.sum(diff ** 2).item()
            norm_sum += torch.sum(p_b.detach().cpu() ** 2).item()

    l2 = np.sqrt(sq_sum)
    rel = l2 / (np.sqrt(norm_sum) + 1e-12)
    return l2, rel




# ============================================================
# 7. METHOD DISPATCHER
# This function maps method names in CONFIG['methods'] to their implementation.
# ============================================================

def run_one_method(method_name, ctx, config):
    # Dispatch one method name to its corresponding evaluation/adaptation routine.
    # All methods receive the same context and common hyperparameters.
    # Shared arguments passed to every adaptive method.
    common = dict(
        task_lagged_base_test=ctx["task_lagged"]["base_test"],
        personalized_predictors=ctx["base_predictors"],
        device=ctx["device"],
        pred_len=ctx["pred_len"],
        theta_override=ctx["theta_shared"],
        warmup_windows=ctx["warmup_windows"],
        rmse_threshold_fallback=config["rmse_threshold_fallback"],
        persistence_M=config["persistence_M"],
        R=config["R"],
        E=config["E"],
        lr=config["cl_lr"],
        batch_size=config["batch_size"],
    )

    # Select target-specific regularization/distillation parameters.
    target_col = ctx["target_col"]

    if target_col == "PM2.5":
        lambda_lwf = config["target_regularization"]["PM2.5"]["lambda_lwf"]
        lambda_si = config["target_regularization"]["PM2.5"]["lambda_si"]

    elif target_col == "WSPM":
        lambda_lwf = config["target_regularization"]["WSPM"]["lambda_lwf"]
        lambda_si = config["target_regularization"]["WSPM"]["lambda_si"]

    elif target_col == "TEMP":
        lambda_lwf = config["target_regularization"]["TEMP"]["lambda_lwf"]
        lambda_si = config["target_regularization"]["TEMP"]["lambda_si"]

    else:
        raise ValueError(f"No regularization parameters defined for target: {target_col}")
        
        
    # Measure wall-clock runtime for the selected method.
    start_time = time.perf_counter()
 
    
    # Static baseline: evaluate without any stream-time updates.
    if method_name == "Static":
        result = evaluate_static_stream_rmse(
            models_by_client=ctx["base_predictors"],
            task_lagged_base_test=ctx["task_lagged"]["base_test"],
            device=ctx["device"],
            pred_len=ctx["pred_len"],
            warmup_windows=ctx["warmup_windows"],
        )

    # Naive CL: adapt using recent stream samples only.
    elif method_name == "Naive":
        result = NaiveContinualLearning(
            **common,
            recent_buffer_capacity=config["persistence_M"] * ctx["pred_len"],
        )

    # NormalReplay: use a single FIFO memory mixing base and stream samples.
    elif method_name == "NormalReplay":
        result = ContinualLearning_NormalReplay(
            **common,
            initial_replay_buffers=ctx["normal_replay_buffers"],
            replay_fraction=config["replay_fraction"],
            replay_min_samples=config["replay_min_samples"],
            replay_max_samples=config["replay_max_samples"],
            lambda_current=config["lambda_current"],
            lambda_replay=config["lambda_replay"],
            adapt_all_clients_on_global_drift=True,
        )

    # TwoTierReplay: preserve fixed base memory and separate recent online samples.
    elif method_name == "TwoTierReplay":
        result = ContinualLearning_TwoTierReplay(
            **common,
            initial_replay_buffers=ctx["client_replay_buffers"],
            p_offline=config["p_offline"],
            replay_fraction=config["replay_fraction"],
            replay_min_samples=config["replay_min_samples"],
            replay_max_samples=config["replay_max_samples"],
            lambda_current=config["lambda_current"],
            lambda_replay=config["lambda_replay"],
            adapt_all_clients_on_global_drift=True,
        )

    # EWC: protect base-task parameters using Fisher-weighted regularization.
    elif method_name == "EWC":
        result = EWCContinualLearning(
            **common,
            ewc_fishers_by_client=ctx["ewc_fishers_by_client"],
            ewc_ref_params_by_client=ctx["ewc_ref_params_by_client"],
            recent_buffer_capacity=config["persistence_M"] * ctx["pred_len"],
            lambda_ewc=config["lambda_ewc"],
        )

    # LwF: preserve previous behavior through teacher-student prediction matching.
    elif method_name == "LwF":
        result = LwFContinualLearning(
            **common,
            recent_buffer_capacity=config["persistence_M"] * ctx["pred_len"],
            lambda_lwf=lambda_lwf,
        )
        

    # SI: protect parameters according to path-integral importance weights.
    elif method_name == "SI":
        result = SIContinualLearning(
            **common,
            si_omegas_by_client=ctx["si_omegas_by_client"],
            si_ref_params_by_client=ctx["si_ref_params_by_client"],
            recent_buffer_capacity=config["persistence_M"] * ctx["pred_len"],
            lambda_si=lambda_si,
        )




    else:
        raise ValueError(f"Unknown method name: {method_name}")

    end_time = time.perf_counter()
    cpu_time = end_time - start_time

    return result, cpu_time




# ============================================================
# 8. RUN METHODS FOR ONE CONTEXT
# Runs all methods for a fixed target, prediction horizon, and seed.
# ============================================================

def run_all_methods_for_context(ctx, config):
    # Run every configured method for one target/horizon/seed context.
    # Partial CSV files are saved after each method to protect long experiments from data loss.
    base_reference = compute_base_reference(ctx)

    summary_rows = []
    per_client_rows = []

    for method_name in config["methods"]:
        print("\n" + "=" * 100)
        print(f"Running {method_name} | target={ctx['target_col']} | horizon={ctx['pred_len']} | seed={ctx['seed']}")
        print("=" * 100)

        result, cpu_time = run_one_method(
            method_name=method_name,
            ctx=ctx,
            config=config
        )

        row, client_rows = evaluate_method_result(
            method_name=method_name,
            result=result,
            ctx=ctx,
            cpu_time=cpu_time,
            base_reference=base_reference
        )

        summary_rows.append(row)
        per_client_rows.extend(client_rows)

        partial_summary = pd.DataFrame(summary_rows)
        partial_summary = add_stability_plasticity_score(
            partial_summary,
            stream_col="stream_rmse",
            af_base_col="af_base_retain",
            alpha_base=1.0,
        )

        partial_per_client = pd.DataFrame(per_client_rows)

        partial_summary.to_csv(
            os.path.join(
                config["results_dir"],
                f"partial_summary_{ctx['target_col']}_p{ctx['pred_len']}_seed{ctx['seed']}.csv"
            ),
            index=False
        )

        partial_per_client.to_csv(
            os.path.join(
                config["results_dir"],
                f"partial_per_client_{ctx['target_col']}_p{ctx['pred_len']}_seed{ctx['seed']}.csv"
            ),
            index=False
        )

    summary_df = pd.DataFrame(summary_rows)
    summary_df = add_stability_plasticity_score(
        summary_df,
        stream_col="stream_rmse",
        af_base_col="af_base_retain",
        alpha_base=1.0,
    )

    per_client_df = pd.DataFrame(per_client_rows)

    summary_df.to_csv(
        os.path.join(
            config["results_dir"],
            f"summary_{ctx['target_col']}_p{ctx['pred_len']}_seed{ctx['seed']}.csv"
        ),
        index=False
    )

    return summary_df, per_client_df



# ============================================================
# 9. RUN FULL GRID
# Outer experiment loop over all seeds, targets, and horizons.
# ============================================================

def run_full_grid(config):
    # Execute the full experimental grid over seeds, targets, and prediction horizons.
    # The dataset and base-only normalization bounds are computed once and reused.
    set_all_seeds(config["seeds"][0])

    print("\nLoading and preprocessing dataset once...")
    client_dfs, client_names = load_and_preprocess_clients(config)
    global_bounds = compute_global_bounds_base_only(client_dfs, config)

    all_summary = []
    all_per_client = []

    for seed in config["seeds"]:
        for target_col in config["targets"]:
            for pred_len in config["horizons"]:

                ctx = build_context_for_experiment(
                    target_col=target_col,
                    pred_len=pred_len,
                    seed=seed,
                    client_dfs=client_dfs,
                    client_names=client_names,
                    global_bounds=global_bounds,
                    config=config
                )
                

                summary_df, per_client_df = run_all_methods_for_context(
                    ctx=ctx,
                    config=config
                )

                
                
                all_summary.append(summary_df)
                all_per_client.append(per_client_df)

                final_summary_so_far = pd.concat(all_summary, ignore_index=True)
                final_per_client_so_far = pd.concat(all_per_client, ignore_index=True)

                # Continuously checkpoint accumulated grid results.
                final_summary_so_far.to_csv(
                    os.path.join(config["results_dir"], "all_summary_results.csv"),
                    index=False
                )

                final_per_client_so_far.to_csv(
                    os.path.join(config["results_dir"], "all_per_client_results.csv"),
                    index=False
                )

    final_summary = pd.concat(all_summary, ignore_index=True)
    final_per_client = pd.concat(all_per_client, ignore_index=True)

    return final_summary, final_per_client


# ============================================================
# 10. PAPER SUMMARY TABLE
# Converts raw experimental outputs into compact tables for reporting.
# ============================================================

def make_paper_summary_table(final_summary_df):
    # Aggregate raw seed-level results into mean/std tables suitable for reporting in the paper.
    final_summary_df = add_stability_plasticity_score(
        final_summary_df,
        stream_col="stream_rmse",
        af_base_col="af_base_retain",
        alpha_base=1.0,
    )

    numeric_cols = [
        "stream_rmse",
        "base_model_base_rmse",
        "final_model_base_rmse",
        "af_base_retain",
        "stability_plasticity_score",
        "total_drifts",
        "avg_drifts_client",
        "global_drift_events",
        "cpu_sec",
    ]

    summary = (
        final_summary_df
        .groupby(["target", "horizon", "method"])[numeric_cols]
        .agg(["mean", "std"])
        .reset_index()
    )

    summary.columns = [
        "_".join(col).strip("_") if isinstance(col, tuple) else col
        for col in summary.columns
    ]

    return summary


def make_best_method_table(paper_summary_df, include_static=False):
    # Rank methods within each target/horizon setting using the stability-plasticity score.
    df = paper_summary_df.copy()

    if not include_static:
        df = df[df["method"] != "Static"].copy()

    df["rank"] = (
        df.groupby(["target", "horizon"])["stability_plasticity_score_mean"]
        .rank(method="min", ascending=True)
    )

    df = df.sort_values(
        ["target", "horizon", "rank", "stability_plasticity_score_mean"]
    )

    cols = [
        "target",
        "horizon",
        "rank",
        "method",
        "stream_rmse_mean",
        "af_base_retain_mean",
        "stability_plasticity_score_mean",
        "total_drifts_mean",
        "cpu_sec_mean",
    ]

    return df[cols]



def add_stability_plasticity_score(
    df,
    stream_col="stream_rmse",
    af_base_col="af_base_retain",
    alpha_base=1.0,
):
    # Add the signed stability-plasticity metric used as the main trade-off score.
    """
    Adds the signed stability-plasticity score.

    Lower is better.

    stream_rmse:
        Plasticity/adaptation error on the CL stream.

    af_base_retain:
        Signed forgetting on the base period.
        Positive = forgetting.
        Negative = improvement / backward transfer.

    Score:
        stream_rmse + alpha_base * af_base_retain
    """

    df = df.copy()

    df["stability_plasticity_score"] = (
        df[stream_col]
        + alpha_base * df[af_base_col]
    )

    return df



def make_method_ranking_table(final_summary_df):
    # Produce an overall ranking table by averaging method ranks across conditions.
    df = final_summary_df.copy()

    df = add_stability_plasticity_score(
        df,
        stream_col="stream_rmse",
        af_base_col="af_base_retain",
        alpha_base=1.0,
    )

    rank_cols = [
        "stream_rmse",
        "af_base_retain",
        "stability_plasticity_score",
        "cpu_sec",
    ]

    for col in rank_cols:
        df[f"rank_{col}"] = (
            df.groupby(["target", "horizon"])[col]
            .rank(method="average", ascending=True)
        )

    ranking = (
        df.groupby("method")[
            [
                "rank_stream_rmse",
                "rank_af_base_retain",
                "rank_stability_plasticity_score",
                "rank_cpu_sec",
            ]
        ]
        .mean()
        .reset_index()
        .sort_values("rank_stability_plasticity_score")
    )

    return ranking


# ============================================================
# 11. RUN EVERYTHING
# Final execution block: launches the grid and saves the output tables.
# ============================================================

# Start the full experiment. For debugging, reduce CONFIG['seeds'], CONFIG['targets'],
# CONFIG['horizons'], or CONFIG['base_rounds'] before running this line.
final_summary_df, final_per_client_df = run_full_grid(CONFIG)

paper_summary = make_paper_summary_table(final_summary_df)
method_ranking = make_method_ranking_table(final_summary_df)

paper_summary_path = os.path.join(CONFIG["results_dir"], "paper_summary_table.csv")
ranking_path = os.path.join(CONFIG["results_dir"], "method_ranking_table.csv")
excel_path = os.path.join(CONFIG["results_dir"], "FLTA2026_results.xlsx")

# Save publication-oriented summary artifacts.
paper_summary.to_csv(paper_summary_path, index=False)
method_ranking.to_csv(ranking_path, index=False)

# Report the saved output locations at the end of the run.
print("\n✅ ALL EXPERIMENTS COMPLETED")
print(f"Raw summary:      {os.path.join(CONFIG['results_dir'], 'all_summary_results.csv')}")
print(f"Per-client:       {os.path.join(CONFIG['results_dir'], 'all_per_client_results.csv')}")
print(f"Paper summary:    {paper_summary_path}")
print(f"Method ranking:   {ranking_path}")
print(f"Excel workbook:   {excel_path}")

paper_summary